# Text Processing Tools

## Quick notes

### What this covers
- The classic Unix text toolbox: `grep`, `sed`, `awk`, `cut`, `paste`, `join`, `sort`, `uniq`, `tr`, `wc`, `head`, `tail`, `rev`, `fold`, `column`, `comm`, `diff`, `nl`, `expand`.
- Regular expressions (BRE vs ERE), anchors, classes.
- How to combine them into pipelines, with a log-analysis example.
- Every GNU vs BSD difference we hit on macOS, called out where it bites.

### Key points
- `grep` finds lines, `sed` edits lines, `awk` understands columns and does arithmetic.
- Use the lightest tool that works: `cut` for fixed columns, `grep` for matching, `awk` when you need logic or sums.
- `sort` before `uniq`: `uniq` only collapses NEIGHBOURING duplicates.
- `grep -E` (extended regex) gives you `+ ? | ( ) { }` without backslashes. Plain `grep` (BRE) needs `\+` etc.
- Use `LC_ALL=C` for predictable sorting and byte-wise matching (the notebook already does).
- Quote patterns with single quotes so the shell does not touch `$`, `*`, `!`.

### Gotchas (the part worth revising twice)
- **`sed -i`**: BSD/macOS needs an argument, `sed -i '' 's/a/b/' f`; GNU takes none, `sed -i 's/a/b/' f`. On BSD, `sed -i 's/a/b/' f` eats the script as the backup suffix and fails. Portable: `sed -i.bak ... f && rm f.bak`.
- **`sed` `a`, `i`, `c`**: BSD wants `a\` then a real newline then the text. GNU also accepts the one-liner `1a text`, BSD errors.
- **`awk` here is the "one true awk"**, not gawk: `gensub`, `asort`, `asorti`, `strftime`, `systime` are "undefined function" errors; gawk-isms like `PROCINFO` and `BEGINFILE` do not error, they silently do nothing (worse). `length(array)` works here.
- **`grep -P`** (Perl regex) does not exist in macOS BSD grep. Use `-E`, or `awk`.
- **`sort -h`, `-V`** exist on this macOS sort but are not guaranteed everywhere; check `sort --version`/behaviour first.
- **`tac`** does not exist on macOS; `tail -r` is the BSD equivalent. GNU `head -n -1` ("all but the last") is an error on BSD.
- **`column`** on BSD supports `-t`, `-s`, `-c`, `-x` only; no `--table` long options.
- `grep` exit codes: 0 = matched, 1 = no match, 2 = error. Under `set -e` or `pipefail` a harmless "no match" looks like failure.
- `wc` pads numbers with spaces on BSD. Use `tr -d ' '` or arithmetic to normalise.

### When it matters
- Log analysis, CSV wrangling, config edits, quick one-off reports, glue between programs.

### One-line recap example
```bash
awk -F, 'NR>1 {s[$3]+=$4} END {for (d in s) print d, s[d]}' data.csv | sort
```

## 01. Sample data used in this notebook

- We create small fixed files so every output is repeatable.
- `staff.csv` has a header plus six rows. `app.log` has a handful of log lines.
- Everything lives in the private working folder and is removed in the last section.

In [1]:
# setup: a %%bash cell magic that keeps ONE bash session alive between cells
import atexit, os, shutil, subprocess, tempfile, uuid
from IPython.core.magic import register_cell_magic

_WORK = tempfile.mkdtemp(prefix="bashnb_")
atexit.register(shutil.rmtree, _WORK, ignore_errors=True)
_ENV = {k: v for k, v in os.environ.items() if k not in ("CLICOLOR", "CLICOLOR_FORCE", "FORCE_COLOR", "PYDEVD_DISABLE_FILE_VALIDATION")}
_ENV.update(LC_ALL="C", TZ="UTC", HOME=_WORK, PS1="", PS4="+ ")
_session = None

def _start():
    global _session
    _session = subprocess.Popen(["bash", "--noprofile", "--norc"], stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=_WORK, env=_ENV)

def _clean(text):
    return text.replace(os.path.join(_WORK, "script"), "script").replace(_WORK, "<work>")

def _run_persistent(code, timeout=60):
    global _session
    if _session is None or _session.poll() is not None:
        _start()
    token = uuid.uuid4().hex
    path = os.path.join(_WORK, "script")
    with open(path, "w") as fh:
        fh.write(code + "\n")
    _session.stdin.write(". '%s' </dev/null; printf '\\n%s %%d\\n' $?\n" % (path, token))
    _session.stdin.flush()
    out, status = [], None
    import threading
    result = {}
    def reader():
        while True:
            line = _session.stdout.readline()
            if not line:
                result["status"] = None; return
            if line.startswith(token + " "):
                result["status"] = int(line.split()[1]); return
            out.append(line)
    t = threading.Thread(target=reader, daemon=True); t.start(); t.join(timeout)
    os.remove(path)
    if t.is_alive():
        _session.kill(); _session = None
        return "".join(out) + "[timed out]\n", 124
    if result.get("status") is None:       # the shell itself exited (e.g. the cell ran `exit N`)
        code = _session.wait(); _session = None
        return "".join(out) + "(the bash session ended; the next cell starts a fresh one)\n", code
    text = "".join(out)
    if text.endswith("\n\n"):
        text = text[:-1]          # drop the blank line our status marker adds
    return text, result.get("status") if result.get("status") is not None else 1

def _run_fresh(code, timeout=60):
    path = os.path.join(_WORK, "script")
    with open(path, "w") as fh:
        fh.write(code + "\n")
    done = subprocess.run(["bash", "--noprofile", "--norc", path], stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, cwd=_WORK, env=_ENV, timeout=timeout)
    os.remove(path)
    return done.stdout, done.returncode

@register_cell_magic
def bash(line, cell):
    """Run the cell in bash. Flags: fresh (new shell), expect_fail (show non-zero status instead of raising)."""
    flags = line.split()
    text, status = (_run_fresh if "fresh" in flags else _run_persistent)(cell)
    print(_clean(text), end="" if text.endswith("\n") or not text else "\n")
    if status != 0:
        if "expect_fail" in flags:
            print("[exit status %d]" % status)
        else:
            raise RuntimeError("bash cell exited with status %d" % status)

In [2]:
%%bash
mkdir tp && cd tp
cat > staff.csv <<'EOF'
id,name,department,salary
1,Alice,Engineering,95000
2,Bob,Sales,65000
3,Carol,Engineering,105000
4,Dave,Marketing,70000
5,Eve,Sales,72000
6,Frank,Engineering,98000
EOF
cat > app.log <<'EOF'
2024-05-01 10:00:01 INFO  user=alice action=login ms=120
2024-05-01 10:00:07 ERROR user=bob action=upload ms=950 code=500
2024-05-01 10:01:15 INFO  user=carol action=login ms=95
2024-05-01 10:02:30 WARN  user=alice action=download ms=400
2024-05-01 10:03:44 ERROR user=carol action=upload ms=1200 code=503
2024-05-01 10:04:02 INFO  user=bob action=logout ms=30
2024-05-01 10:05:59 ERROR user=alice action=upload ms=870 code=500
EOF
wc -l staff.csv app.log

       7 staff.csv
       7 app.log
      14 total


## 02. `grep` basics: `-i -v -n -c`

- `grep PATTERN file` prints matching lines.
- `-i` ignore case, `-v` invert (lines that do NOT match), `-n` show line numbers, `-c` count matching lines.
- `-c` counts lines, not occurrences.
- Combine: `grep -vc` counts the non-matching lines.

In [3]:
%%bash
echo "--- match:"; grep Engineering staff.csv
echo "--- case-insensitive (-i):"; grep -ic engineering staff.csv
echo "--- inverted count (-vc):"; grep -vc Engineering staff.csv
echo "--- with line numbers (-n):"; grep -n Sales staff.csv
echo "--- several patterns (-e):"; grep -e Bob -e Eve staff.csv

--- match:
1,Alice,Engineering,95000
3,Carol,Engineering,105000
6,Frank,Engineering,98000
--- case-insensitive (-i):
3
--- inverted count (-vc):
4
--- with line numbers (-n):
3:2,Bob,Sales,65000
6:5,Eve,Sales,72000
--- several patterns (-e):
2,Bob,Sales,65000
5,Eve,Sales,72000


## 03. `grep` more: `-o -w -F -E -x -m -q`

- `-o` prints only the matching part (one per line).
- `-w` match whole words only. `-x` the whole line must match.
- `-F` fixed string: no regex, dots and stars mean themselves (often faster, and safer for user input).
- `-E` extended regex. `-m N` stop after N matches. `-q` print nothing, just set the exit status (great in `if`).

In [4]:
%%bash
echo "--- only the matched numbers (-o):"; grep -oE '[0-9]+' app.log | head -4
echo "--- whole word only (-ow) skips 'username':"; echo "username user" | grep -ow user
echo "--- regex dot vs fixed string:"
echo "axb" | grep 'a.b'
echo "axb" | grep -F 'a.b' || echo "(-F treats the dot literally: no match)"
echo "--- first two matches only (-m2):"; grep -m2 INFO app.log
echo "--- quiet test in an if:"
if grep -q ERROR app.log; then echo "errors present"; fi
echo "--- whole line (-x):"; echo "ab" | grep -x a || echo "no whole-line match"

--- only the matched numbers (-o):
2024
05
01
10
--- whole word only (-ow) skips 'username':
user
--- regex dot vs fixed string:
axb
(-F treats the dot literally: no match)
--- first two matches only (-m2):
2024-05-01 10:00:01 INFO  user=alice action=login ms=120
2024-05-01 10:01:15 INFO  user=carol action=login ms=95
--- quiet test in an if:
errors present
--- whole line (-x):
no whole-line match


## 04. `grep` across files: `-r -l -h -A -B -C --include`

- `-r` recurse into directories. `-l` list only file names with a match; `-L` the opposite.
- `-h` hide file names; `-H` force them (default when more than one file).
- `-A n`, `-B n`, `-C n`: show n lines After / Before / around each match. Separator lines `--` appear between groups.
- `--include='*.txt'` / `--exclude-dir=sub` filter what `-r` visits (both work on macOS grep).
- `sort` the output of `-r` if you need stable order; directory order is not guaranteed.

In [5]:
%%bash
mkdir -p tree/sub
echo "hello world" > tree/a.txt
echo "hello again" > tree/sub/b.log
echo "nothing"     > tree/c.txt
echo "--- -r with names:"; grep -r hello tree | sort
echo "--- -rl (files only):"; grep -rl hello tree | sort
echo "--- -rh (no names):"; grep -rh hello tree | sort
echo "--- --include:"; grep -r --include='*.txt' hello tree
echo "--- --exclude-dir:"; grep -r --exclude-dir=sub hello tree
echo "--- -rc (count per file):"; grep -rc hello tree | sort
echo "--- context: -B1 -A1 around the ERROR on line 5:"
grep -n -B1 -A1 'code=503' app.log

--- -r with names:
tree/a.txt:hello world
tree/sub/b.log:hello again
--- -rl (files only):
tree/a.txt
tree/sub/b.log
--- -rh (no names):
hello again
hello world
--- --include:
tree/a.txt:hello world
--- --exclude-dir:
tree/a.txt:hello world
--- -rc (count per file):
tree/a.txt:1
tree/c.txt:0
tree/sub/b.log:1
--- context: -B1 -A1 around the ERROR on line 5:
4-2024-05-01 10:02:30 WARN  user=alice action=download ms=400
5:2024-05-01 10:03:44 ERROR user=carol action=upload ms=1200 code=503
6-2024-05-01 10:04:02 INFO  user=bob action=logout ms=30


## 05. `grep` exit statuses

- `0`: at least one line matched. `1`: no line matched (NOT an error). `2`: real error, such as a missing file.
- `-s` silences "file not found" messages but the status stays 2.
- In scripts under `set -e`, guard greps that may legitimately find nothing: `grep ... || true`.

In [6]:
%%bash expect_fail
grep -q ERROR app.log;        echo "match -> $?"
grep -q NOTHING app.log;      echo "no match -> $?"
grep ERROR missing.log;       echo "missing file -> $?"
grep -s ERROR missing.log;    echo "missing, -s -> $?"

match -> 0
no match -> 1
grep: missing.log: No such file or directory
missing file -> 2
missing, -s -> 2


## 06. Regex cheat sheet: BRE vs ERE

- **Anchors**: `^` start of line, `$` end of line.
- **Any char**: `.`; **repeat**: `*` zero or more; ERE adds `+` (one or more), `?` (optional) and `{n,m}` as plain characters (BRE intervals need backslashes, see below).
- **Classes**: `[abc]`, `[^abc]`, ranges `[a-z]`, and POSIX classes `[[:digit:]]`, `[[:alpha:]]`, `[[:space:]]`, `[[:upper:]]`.
- **Groups / alternation**: ERE `(a|b)`; BRE needs `\(a\)` and `\|`. Interval in BRE is `\{2\}`.
- **Word boundaries**: BSD grep accepts `\<` `\>` and `\b`; they are not POSIX, so prefer `-w` for portability.
- `\d` happened to work in BSD `grep -E` here, but it is not standard (on GNU/Linux `grep -E` does not treat it as "digit" (not run here)); use `[[:digit:]]`.
- `egrep` is the old name for `grep -E`; prefer `grep -E`.

In [7]:
%%bash
echo "--- anchors:"
printf 'start here\nnot start\n' | grep '^start'
printf 'ends with x\nx then more\n' | grep 'x$'
echo "--- BRE needs backslashes, ERE does not (same result):"
echo "color colour" | grep -o 'colou\?r'
echo "color colour" | grep -oE 'colou?r'
echo "--- alternation:"
printf 'cat\ndog\nbird\n' | grep -E '^(cat|dog)$'
echo "--- intervals:"
printf 'ab\naab\naaab\n' | grep -E '^a{2,3}b$'
echo "--- POSIX class:"
echo "id42x" | grep -oE '[[:digit:]]+'
echo "--- word boundary vs -w:"
echo "foo bar" | grep '\<bar\>'
echo "foo bar" | grep -w bar

--- anchors:
start here
ends with x
--- BRE needs backslashes, ERE does not (same result):
color
colour
color
colour
--- alternation:
cat
dog
--- intervals:
aab
aaab
--- POSIX class:
42
--- word boundary vs -w:
foo bar
foo bar


## 07. `sed`: substitution `s///` and flags

- `s/old/new/` replaces the FIRST match on each line. Flags: `g` all matches, `N` the Nth match, `p` print, `I` ignore case (BSD sed here accepts `I`).
- Any delimiter works: `s|/usr/bin|/opt/bin|` avoids escaping slashes.
- `&` in the replacement means "the whole match". Escape a literal one as `\&`.
- Groups: BRE `\( \)` with `\1`; with `-E`, plain `( )` and `\1`.
- `\n` in a replacement does produce a newline on this macOS sed, but it is not guaranteed on every sed; the portable way is a backslash followed by a real newline (used below).
- sed is line-based; it reads stdin or files and writes stdout (the file is untouched without `-i`).

In [8]:
%%bash
echo "one two one two" | sed 's/one/1/'
echo "one two one two" | sed 's/one/1/g'
echo "one two one two" | sed 's/one/1/2'
echo "Hello hello" | sed 's/hello/X/I'
echo "/usr/bin/tool" | sed 's|/usr/bin|/opt/bin|'
echo "price 5" | sed 's/[0-9]/[&]/'
echo "a & b" | sed 's/&/and/'
echo "x" | sed 's/x/a\&b/'
echo "Alice Smith" | sed 's/\(.*\) \(.*\)/\2, \1/'
echo "Alice Smith" | sed -E 's/(.*) (.*)/\2, \1/'
echo "a b" | sed 's/ /\
/'

1 two one two
1 two 1 two
one two 1 two
X hello
/opt/bin/tool
price [5]
a and b
a&b
Smith, Alice
Smith, Alice
a
b


## 08. `sed`: addresses, ranges, `-n`, `p`, `d`, `y`, `-e`

- An address picks lines: a number `3`, a range `2,4`, last line `$`, a regex `/ERROR/`, or a range of patterns `/start/,/end/`.
- `-n` turns off auto-printing; `p` prints. So `sed -n '2,3p'` prints lines 2-3.
- `d` deletes the line; `q` quits (like `head`); `!` negates: `2!d` deletes every line except 2.
- `y/abc/xyz/` transliterates characters (like `tr`).
- `-e` chains several commands; `;` separates them too. Inside `{ }` end the last command with `;` before the `}` (as below).

In [9]:
%%bash
echo "--- lines 2-3:"; sed -n '2,3p' staff.csv
echo "--- last line:"; sed -n '$p' staff.csv
echo "--- delete header:"; sed '1d' staff.csv | head -2
echo "--- only lines matching:"; sed -n '/Sales/p' staff.csv
echo "--- pattern range:"; sed -n '/Bob/,/Dave/p' staff.csv
echo "--- keep only line 1 (1!d deletes every other line):"; sed '1!d' staff.csv
echo "--- stop after 2 lines (q):"; sed 2q staff.csv
echo "--- y transliterate:"; echo hello | sed 'y/el/ip/'
echo "--- several commands (-e):"; sed -n -e '1p' -e '$p' staff.csv
echo "--- run a block on matching lines:"; sed -n '/Sales/{s/Sales/S/;p;}' staff.csv

--- lines 2-3:
1,Alice,Engineering,95000
2,Bob,Sales,65000
--- last line:
6,Frank,Engineering,98000
--- delete header:
1,Alice,Engineering,95000
2,Bob,Sales,65000
--- only lines matching:
2,Bob,Sales,65000
5,Eve,Sales,72000
--- pattern range:
2,Bob,Sales,65000
3,Carol,Engineering,105000
4,Dave,Marketing,70000
--- keep only line 1 (1!d deletes every other line):
id,name,department,salary
--- stop after 2 lines (q):
id,name,department,salary
1,Alice,Engineering,95000
--- y transliterate:
hippo
--- several commands (-e):
id,name,department,salary
6,Frank,Engineering,98000
--- run a block on matching lines:
2,Bob,S,65000
5,Eve,S,72000


## 09. `sed`: append, insert, change (the BSD newline rule)

- `a\` append after, `i\` insert before, `c\` replace the line. The text goes on the NEXT line.
- GNU sed also accepts `1a text` on one line. **BSD sed errors** (shown below).
- Portable form: `a\` followed by a real newline, then the text. In a script file, or inside quotes with an actual newline.
- To add several lines, end each line except the last with `\`.

In [10]:
%%bash
echo "--- append after line 1:"
printf 'a\nb\n' | sed '1a\
APPENDED'
echo "--- insert before line 2:"
printf 'a\nb\n' | sed '2i\
INSERTED'
echo "--- change line 2:"
printf 'a\nb\nc\n' | sed '2c\
CHANGED'
echo "--- two lines in one go:"
printf 'a\nb\n' | sed '1a\
first extra\
second extra'

--- append after line 1:
a
APPENDED
b
--- insert before line 2:
a
INSERTED
b
--- change line 2:
a
CHANGED
c
--- two lines in one go:
a
first extra
second extra
b


In [11]:
%%bash expect_fail
# GNU-style one-liner: fails on BSD sed
printf 'a\nb\n' | sed '1a APPENDED'

sed: 1: "1a APPENDED
": command a expects \ followed by text
[exit status 1]


## 10. `sed -i`: editing files in place, portably

- GNU: `sed -i 's/a/b/' f`. BSD/macOS: `sed -i '' 's/a/b/' f` (empty string = no backup).
- On BSD, `sed -i 's/a/b/' f` takes `s/a/b/` as the backup suffix and `f` as the script: error.
- The portable trick: `sed -i.bak 's/a/b/' f` works on BOTH (suffix attached to `-i`), then delete the `.bak`.
- Or avoid `-i`: `sed ... f > f.tmp && mv f.tmp f`.

In [12]:
%%bash
printf 'cat\ndog\n' > pets.txt
sed -i '' 's/cat/CAT/' pets.txt          # BSD form
cat pets.txt
sed -i.bak 's/dog/DOG/' pets.txt         # works on BSD and GNU
echo "--- after -i.bak, files:"; ls pets.txt*
cat pets.txt
rm pets.txt.bak
echo "--- no-flag way:"
sed 's/CAT/Cat/' pets.txt > pets.tmp && mv pets.tmp pets.txt
cat pets.txt

CAT
dog
--- after -i.bak, files:
pets.txt
pets.txt.bak
CAT
DOG
--- no-flag way:
Cat
DOG


In [13]:
%%bash expect_fail
# GNU form on BSD sed: 's/DOG/Dog/' is taken as the backup suffix and 'pets.txt' is then read as the script
sed -i 's/DOG/Dog/' pets.txt </dev/null
echo "file unchanged: $(tr '\n' ' ' < pets.txt)"

sed: 1: "pets.txt
": extra characters at the end of p command
file unchanged: Cat DOG 


## 11. `awk`: fields, `-F`, `NR`, `NF`, `FNR`

- awk splits each line into fields `$1 $2 ...`; `$0` is the whole line, `$NF` the last field.
- Default separator is runs of blanks. `-F,` sets comma; `-F'\t'` tab; `-F'[:;]'` a regex.
- `NR` is the line number so far, `NF` the number of fields, `FNR` the line number within the current file.
- Assigning to a field (`$1=$1`) makes awk rebuild the line with `OFS` (default one space).

In [14]:
%%bash
echo "--- names (skip header with NR>1):"
awk -F, 'NR>1 {print $2}' staff.csv
echo "--- last field, number of fields:"
echo "a b c d" | awk '{print NF, $NF, $(NF-1)}'
echo "--- rebuild a line to squeeze spaces:"
echo "a    b   c" | awk '{$1=$1} 1'
echo "--- regex field separator:"
echo "k1:v1;k2:v2" | awk -F'[:;]' '{print $2, $4}'
echo "--- OFS for the output:"
echo "a b c" | awk 'BEGIN{OFS="-"} {$1=$1; print}'

--- names (skip header with NR>1):
Alice
Bob
Carol
Dave
Eve
Frank
--- last field, number of fields:
4 d c
--- rebuild a line to squeeze spaces:
a b c
--- regex field separator:
v1 v2
--- OFS for the output:
a-b-c


## 12. `awk`: patterns, `BEGIN`, `END`

- An awk program is `pattern { action }` pairs. No pattern = every line. No action = print the line.
- Patterns: `NR>1`, `$3=="Sales"`, `/regex/`, `$4>70000`, `NR==2,NR==4` (range), `!seen[$0]++` (unique).
- `BEGIN { }` runs once before input (set `FS`, print headers); `END { }` once after (totals).
- Inside `END`, `NR` is the total line count. `$0` still holds the last line in this awk, but do not rely on that in other awks.

In [15]:
%%bash
echo "--- pattern only (no action):"; awk -F, '$3=="Sales"' staff.csv
echo "--- numeric comparison:"; awk -F, 'NR>1 && $4>90000 {print $2, $4}' staff.csv
echo "--- regex pattern on a field:"; awk -F, '$2 ~ /^[A-C]/ {print $2}' staff.csv
echo "--- BEGIN and END:"
awk -F, 'BEGIN {print "name,salary"} NR>1 {n++; t+=$4} END {print "rows="n, "total="t, "avg="t/n}' staff.csv
echo "--- unique lines, keep first (no sort needed):"
printf 'b\na\nb\nc\na\n' | awk '!seen[$0]++'
echo "--- every 2nd line:"
printf '1\n2\n3\n4\n' | awk 'NR%2==0'

--- pattern only (no action):
2,Bob,Sales,65000
5,Eve,Sales,72000
--- numeric comparison:
Alice 95000
Carol 105000
Frank 98000
--- regex pattern on a field:
Alice
Bob
Carol
--- BEGIN and END:
name,salary
rows=6 total=505000 avg=84166.7
--- unique lines, keep first (no sort needed):
b
a
c
--- every 2nd line:
2
4


## 13. `awk`: arrays, `printf`, `-v`, group-by

- awk arrays are associative: `count[$3]++`. Keys are strings. Check with `k in arr`; remove with `delete arr[k]`.
- `for (k in arr)` has NO guaranteed order, so pipe through `sort` for stable output.
- `printf` works like C: `%s %d %5.2f %-10s %05d %x`. It does not add a newline.
- `-v name=value` passes a shell value in safely (do NOT splice shell variables into the program text). awk still processes backslash escapes in the value.

In [16]:
%%bash
echo "--- salary per department (group-by sum), sorted:"
awk -F, 'NR>1 {sum[$3]+=$4; n[$3]++} END {for (d in sum) printf "%-12s %7d %3d\n", d, sum[d], n[d]}' staff.csv | sort
echo "--- shell value in with -v:"
min=70000
awk -F, -v min="$min" 'NR>1 && $4>=min {print $2}' staff.csv
echo "--- printf formats:"
awk 'BEGIN {printf "%-6s|%6.2f|%05d|%x|%c\n", "ab", 3.14159, 42, 255, 65}'
echo "--- membership and delete:"
awk 'BEGIN {a["x"]=1; if ("x" in a) print "x is in"; delete a["x"]; print "size now", length(a)}'

--- salary per department (group-by sum), sorted:
Engineering   298000   3
Marketing      70000   1
Sales         137000   2
--- shell value in with -v:
Alice
Carol
Dave
Eve
Frank
--- printf formats:
ab    |  3.14|00042|ff|A
--- membership and delete:
x is in
size now 0


## 14. `awk`: built-in functions and one-true-awk limits

- Available here: `length() substr() index() split() sub() gsub() match() sprintf() tolower() toupper() int() sqrt() system()`.
- `substr(s, start, len)` is 1-based. Passing 0 as start behaves differently between awks (here it returned two characters for len 2), so avoid it.
- `sub`/`gsub` edit a variable in place and return the count.
- NOT in macOS awk: `gensub`, `asort`, `asorti`, `strftime`, `systime` (calling them is a runtime error, `gensub` shown below). `PROCINFO` and `BEGINFILE` do not error but do nothing.
- Need sorted output? Pipe to `sort` rather than relying on awk extensions.

In [17]:
%%bash
awk 'BEGIN {
  s = "Hello, World"
  print length(s), toupper(s), tolower(s)
  print substr(s, 8, 5), index(s, "World")
  n = split("a:b:c", p, ":"); print n, p[1], p[3]
  t = s; c = gsub(/o/, "0", t); print c, t
  if (match(s, /W[a-z]+/)) print RSTART, RLENGTH
  print sprintf("%03d", 7), int(7.9), int(-7.9), 7 % 3, 2 ^ 10
}'
echo "--- regex interval works in this awk:"
echo "aaa" | awk '/a{3}/ {print "matches a{3}"}'

12 HELLO, WORLD hello, world
World 8
3 a c
2 Hell0, W0rld
8 5
007 7 -7 1 1024
--- regex interval works in this awk:
matches a{3}


In [18]:
%%bash expect_fail
awk 'BEGIN { print gensub(/a/, "b", "g", "aaa") }'

awk: calling undefined function gensub
 source line number 1
[exit status 2]


## 15. `awk` with several files: `NR==FNR`

- `NR` counts across all files, `FNR` restarts per file. So `NR==FNR` is true only while reading the FIRST file.
- Idiom: load file 1 into an array, then use it while reading file 2 (a join / lookup).
- Redirect output per key with parentheses: `print > ("out_" $1 ".txt")`. Without the parentheses, one-true-awk gives a syntax error.

In [19]:
%%bash
printf 'Alice,Engineering\nBob,Sales\n' > depts.txt
printf 'Alice,95000\nBob,65000\nZed,1\n' > pay.txt
echo "--- lookup department for each pay row:"
awk -F, 'NR==FNR {dept[$1]=$2; next} {print $1, $2, ($1 in dept ? dept[$1] : "unknown")}' depts.txt pay.txt
echo "--- FNR vs NR:"
awk 'FNR==1 {print FILENAME ": starts at overall line " NR}' depts.txt pay.txt
echo "--- split into files by key:"
awk -F, 'NR>1 {print $2 > ("dept_" $3 ".txt")}' staff.csv
ls dept_*.txt; cat dept_Sales.txt

--- lookup department for each pay row:
Alice 95000 Engineering
Bob 65000 Sales
Zed 1 unknown
--- FNR vs NR:
depts.txt: starts at overall line 1
pay.txt: starts at overall line 3
--- split into files by key:
dept_Engineering.txt
dept_Marketing.txt
dept_Sales.txt
Bob
Eve


## 16. `cut`, `paste`, `join`

- `cut -d, -f1,3` picks fields; `-f2-` from field 2 on; `-c2-4` picks characters. One-char delimiters only; no quoting awareness.
- `cut` cannot reorder fields (`-f3,1` still prints 1 then 3). Use awk to reorder.
- `paste file1 file2` glues lines side by side (tab); `-d` sets the delimiter; `-s` makes one line from a file.
- `join` merges two files on a key field. Both inputs MUST be sorted on that key (use `<(sort ...)`). Use `-t,` for CSV.

In [20]:
%%bash
echo "--- cut:"
cut -d, -f2,4 staff.csv | head -3
cut -d, -f3- staff.csv | head -2
echo abcdef | cut -c2-4
echo "--- cut cannot reorder; awk can:"
echo "a,b,c" | cut -d, -f3,1
echo "a,b,c" | awk -F, '{print $3 "," $1}'
echo "--- paste:"
printf '1\n2\n' > nums.txt; printf 'a\nb\n' > lets.txt
paste -d, nums.txt lets.txt
paste -s -d'+' nums.txt
echo "--- join on the first field:"
printf 'a,1\nb,2\n' > j1.txt; printf 'b,x\na,y\n' > j2.txt
join -t, <(sort j1.txt) <(sort j2.txt)

--- cut:
name,salary
Alice,95000
Bob,65000
department,salary
Engineering,95000
bcd
--- cut cannot reorder; awk can:
a,c
c,a
--- paste:
1,a
2,b
1+2
--- join on the first field:
a,1,y
b,2,x


## 17. `sort` and `uniq`

- `sort` orders lines (bytewise under `LC_ALL=C`). `-n` numeric, `-r` reverse, `-u` unique, `-s` stable, `-f` fold case.
- `-t,` sets the field separator; `-k2,2` sorts on field 2 only. `-k2` (no end) uses field 2 to the END of the line. Add flags per key: `-k2,2nr`.
- Without `-n`, `10` sorts before `9`.
- `-h` (human sizes like 2K, 1M) and `-V` (version order) work on this macOS sort, but older systems may lack them (not run here).
- `uniq -c` prefixes counts; `uniq -d` shows only duplicated lines; `uniq -u` only lines that occur once. Always `sort` first.

In [21]:
%%bash
echo "--- default sort puts 10 before 9:"; printf '9\n10\n2\n' | sort
echo "--- numeric (-n):"; printf '9\n10\n2\n' | sort -n
echo "--- reverse numeric by salary column:"; tail -n +2 staff.csv | sort -t, -k4,4nr | head -3
echo "--- sort by dept then name:"; tail -n +2 staff.csv | sort -t, -k3,3 -k2,2 | cut -d, -f2,3
echo "--- unique (-u):"; printf 'b\na\nb\n' | sort -u
echo "--- sizes with -h:"; printf '1M\n2K\n3G\n' | sort -h
echo "--- version sort -V:"; printf 'v1.10\nv1.2\n' | sort -V
echo "--- uniq -c (sort first!):"; tail -n +2 staff.csv | cut -d, -f3 | sort | uniq -c
echo "--- uniq without sort misses non-adjacent dups:"; printf 'a\nb\na\n' | uniq | tr '\n' ' '; echo
echo "--- only duplicated (-d):"; printf 'a\nb\na\n' | sort | uniq -d

--- default sort puts 10 before 9:
10
2
9
--- numeric (-n):
2
9
10
--- reverse numeric by salary column:
3,Carol,Engineering,105000
6,Frank,Engineering,98000
1,Alice,Engineering,95000
--- sort by dept then name:
Alice,Engineering
Carol,Engineering
Frank,Engineering
Dave,Marketing
Bob,Sales
Eve,Sales
--- unique (-u):
a
b
--- sizes with -h:
2K
1M
3G
--- version sort -V:
v1.2
v1.10
--- uniq -c (sort first!):
   3 Engineering
   1 Marketing
   2 Sales
--- uniq without sort misses non-adjacent dups:
a b a 
--- only duplicated (-d):
a


## 18. `tr`, `wc`, `head`, `tail`, `rev`, `fold`

- `tr SET1 SET2` translates characters; `-d` deletes; `-s` squeezes repeats; classes like `[:lower:]` work. `tr` reads stdin only and works on single characters, not words.
- `wc -l/-w/-c` counts lines/words/bytes. BSD `wc` pads with spaces.
- `head -n N`, `tail -n N`; `tail -n +2` means "from line 2 on" (drop the header). GNU `head -n -1` ("all but last") is an error on BSD.
- `tail -r` reverses line order on BSD (GNU has `tac`, macOS does not).
- `rev` reverses each line; `fold -w N` wraps at N characters.

In [22]:
%%bash
echo "hello world" | tr 'a-z' 'A-Z'
echo "hello world" | tr '[:lower:]' '[:upper:]'
echo "h-e-l-l-o" | tr -d '-'
echo "a    b     c" | tr -s ' '
printf 'a\nb\nc\n' | tr '\n' ','; echo
echo "--- wc (BSD pads, strip it):"; wc -l < staff.csv | tr -d ' '
echo "--- header-less data with tail -n +2:"; tail -n +2 staff.csv | head -2
echo "--- last two lines:"; tail -n 2 app.log | cut -c1-19
echo "--- reverse line order (BSD tail -r):"; printf 'a\nb\nc\n' | tail -r | tr '\n' ' '; echo
echo "--- rev and fold:"; echo abc | rev; echo abcdefgh | fold -w 3
echo "--- tac on this machine?"; command -v tac >/dev/null && echo yes || echo "no tac (use tail -r)"

HELLO WORLD
HELLO WORLD
hello
a b c
a,b,c,
--- wc (BSD pads, strip it):
7
--- header-less data with tail -n +2:
1,Alice,Engineering,95000
2,Bob,Sales,65000
--- last two lines:
2024-05-01 10:04:02
2024-05-01 10:05:59
--- reverse line order (BSD tail -r):
c b a 
--- rev and fold:
cba
abc
def
gh
--- tac on this machine?
no tac (use tail -r)


In [23]:
%%bash expect_fail
echo "--- GNU 'all but the last line' fails on BSD:"
printf 'a\nb\nc\n' | head -n -1

--- GNU 'all but the last line' fails on BSD:
head: illegal line count -- -1
[exit status 1]


## 19. `column`, `nl`, `expand`, `comm`, `diff`

- `column -t` aligns whitespace-separated text into a neat table; `-s,` sets the input separator. BSD `column` has no long options.
- `nl` numbers lines (`-ba` numbers blank lines too). `cat -n` is similar.
- `expand` converts tabs to spaces (`-t 4` for width 4).
- `comm` compares two SORTED files in 3 columns: only-in-1, only-in-2, both. `-12` shows only the common lines; `-23` only in file 1.
- `diff -u` gives a unified diff (exit status 1 means "differences", not error). Try `diff -q` to just test.

In [24]:
%%bash
echo "--- column -t -s,:"; head -3 staff.csv | column -t -s,
echo "--- nl:"; printf 'x\ny\n' | nl -ba -w2 -s': '
echo "--- expand:"; printf 'a\tb\n' | expand | cat -et
echo "--- comm:"
printf 'a\nb\nc\n' > c1.txt; printf 'b\nc\nd\n' > c2.txt
comm c1.txt c2.txt | cat -et
echo "--- only in c1:"; comm -23 c1.txt c2.txt
echo "--- unified diff (header lines skipped):"
diff -u c1.txt c2.txt | sed 1,2d
echo "status: ${PIPESTATUS[0]}"

--- column -t -s,:
id  name   department   salary
1   Alice  Engineering  95000
2   Bob    Sales        65000
--- nl:
 1: x
 2: y
--- expand:
a       b$
--- comm:
a$
^I^Ib$
^I^Ic$
^Id$
--- only in c1:
a
--- unified diff (header lines skipped):
@@ -1,3 +1,3 @@
-a
 b
 c
+d
status: 1


## 20. Log-analysis pipeline

- Goal: from `app.log` answer typical questions with small pipelines.
- Break each question into: select lines (grep), pick fields (awk/cut), count (sort | uniq -c), rank (sort -nr).
- Build the pipeline stage by stage, checking the output at each step.

In [25]:
%%bash
echo "--- 1. events per level:"
awk '{print $3}' app.log | sort | uniq -c | sort -k1,1nr -k2,2
echo "--- 2. error count per user:"
grep ERROR app.log | sed 's/.*user=\([a-z]*\).*/\1/' | sort | uniq -c | sort -k1,1nr -k2,2
echo "--- 3. average ms for uploads (awk does the math):"
grep 'action=upload' app.log | sed 's/.*ms=\([0-9]*\).*/\1/' | awk '{s+=$1} END {printf "%.1f over %d requests\n", s/NR, NR}'
echo "--- 4. slowest request:"
awk '{for (i=1;i<=NF;i++) if ($i ~ /^ms=/) {split($i, kv, "="); print kv[2], $2, $4}}' app.log | sort -nr | head -1
echo "--- 5. errors with their http code:"
grep -o 'code=[0-9]*' app.log | sort | uniq -c
echo "--- 6. requests per minute:"
cut -c1-16 app.log | sort | uniq -c

--- 1. events per level:
   3 ERROR
   3 INFO
   1 WARN
--- 2. error count per user:
   1 alice
   1 bob
   1 carol
--- 3. average ms for uploads (awk does the math):
1006.7 over 3 requests
--- 4. slowest request:
1200 10:03:44 user=carol
--- 5. errors with their http code:
   2 code=500
   1 code=503
--- 6. requests per minute:
   2 2024-05-01 10:00
   1 2024-05-01 10:01
   1 2024-05-01 10:02
   1 2024-05-01 10:03
   1 2024-05-01 10:04
   1 2024-05-01 10:05


## 21. `awk` vs `jq`, and picking the right tool

- awk, grep, sed work on LINES of text. For JSON use `jq`, which understands structure (nesting, quotes, escapes).
- Never parse JSON with regex or `cut`: one extra space or reordered key breaks it.
- `jq` is not installed everywhere, so guard it: `command -v jq`.
- Rule of thumb: match = grep, replace = sed, columns/sums = awk, JSON = jq, quick column slice = cut.

In [26]:
%%bash
if command -v jq >/dev/null; then
  printf '{"users":[{"name":"alice","ms":120},{"name":"bob","ms":950}]}\n' > data.json
  echo "--- names:"; jq -r '.users[].name' data.json
  echo "--- total ms:"; jq '[.users[].ms] | add' data.json
  echo "--- the same ask with awk is fragile (works only on this exact layout):"
  grep -o '"ms":[0-9]*' data.json | awk -F: '{s+=$2} END {print s}'
else
  echo "skipped: jq not installed"
fi

--- names:
alice
bob
--- total ms:
1070
--- the same ask with awk is fragile (works only on this exact layout):
1070


## 22. Bad vs good, and GNU/BSD differences in one table

| Task | Portable / good | Breaks on one side |
|---|---|---|
| in-place edit | `sed -i.bak ... f; rm f.bak` | `sed -i ...` (BSD), `sed -i '' ...` (GNU) |
| add a line | `sed '1a\` + newline + text | `sed '1a text'` (BSD error) |
| Perl regex | `grep -E`, `awk` | `grep -P` (absent on macOS) |
| reverse lines | `tail -r` (BSD) or `awk '{a[NR]=$0} END{for(i=NR;i>0;i--)print a[i]}'` | `tac` (GNU only) |
| all but last line | `sed '$d'` | `head -n -1` (GNU only) |
| regex gensub/asort | plain `awk` with sub/gsub/split + `sort` | gawk-only functions |
| case-insensitive sed | `sed 's/x/y/I'` works on this BSD sed too | not in strict POSIX |

Common mistakes:
- `cat file | grep x` instead of `grep x file`.
- `grep x file | wc -l` instead of `grep -c x file`.
- Unquoted patterns: `grep a*b file` (if a file matches `a*b`, the shell replaces the pattern with that file name).
- Parsing `ls` output instead of using globs or `find`.

In [27]:
%%bash
echo "--- all but last line, portable:"; printf 'a\nb\nc\n' | sed '$d'
echo "--- reverse lines, portable awk:"; printf 'a\nb\nc\n' | awk '{a[NR]=$0} END {for (i=NR;i>0;i--) print a[i]}' | tr '\n' ' '; echo
echo "--- count the cheap way:"; grep -c ERROR app.log
echo "--- unquoted glob trap:"; touch axb.tmp; echo "aXb" | grep a*b.tmp; echo "(shell expanded a*b.tmp into the file name, so grep got a different pattern)"; rm axb.tmp

--- all but last line, portable:
a
b
--- reverse lines, portable awk:
c b a 
--- count the cheap way:
3
--- unquoted glob trap:
(shell expanded a*b.tmp into the file name, so grep got a different pattern)


## 23. Cleanup

- Remove the sample folder so nothing is left behind.

In [28]:
%%bash
cd ..
rm -rf tp
echo cleaned

cleaned
